### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] !pip install arabic_reshaper

In [ ]:
# [PATCHED] !pip install python-bidi

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
import os
import pandas as pd

working_folder = './ Drive/TransformersCode/03-restaurant/Find_meals/'

csv_file_path = os.path.join(working_folder, 'indian_food.csv')
df = pd.read_csv(csv_file_path)

df.head()

In [ ]:
import numpy as np
similarities_file = working_folder + "cosine_similarity_matrix.npy"

similarities = np.load(similarities_file)
print(f"Similarity matrix loaded from {similarities_file}")
print(similarities.shape)
print(similarities)

In [ ]:
def get_index(recipe_name):
    return df[df.ARname == recipe_name].index[0]

def get_recipe(index):
    return df.iloc[index]['ARname']

def get_recipe_recommendations(recipe_name):

    index = get_index(recipe_name)

    sims = similarities[index]
    scores = list(enumerate(sims))
    print(scores)

    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return sorted_scores

recipe_name = 'دجاج بالزبدة'
sorted_scores=get_recipe_recommendations(recipe_name)
for recipe in sorted_scores[:5]:
    index=recipe[0]
    score=recipe[1]
    recipe=get_recipe(index)
    print(recipe, score)

In [ ]:
def get_recipe_recommendations_from_previous_orders(previous_orders):

    indices = [get_index(recipe_name) for recipe_name in previous_orders]

    sims = similarities[indices]

    avg_similarity = sims.mean(axis=0)

    scores = list(enumerate(avg_similarity))

    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)
    return sorted_scores

previous_orders = ['دجاج بالزبدة', 'دجاج تكا ماسالا']

recommendations = get_recipe_recommendations_from_previous_orders(previous_orders)

top_recommendations = recommendations[:10]

top_indices = [rec[0] for rec in top_recommendations]

top_scores = [rec[1] for rec in top_recommendations]

top_names = [get_recipe(idx) for idx in top_indices]

for i in range(10):
    print(top_names[i])

In [ ]:
import matplotlib.pyplot as plt

import arabic_reshaper
from bidi.algorithm import get_display

top_names_display = [get_display(arabic_reshaper.reshape(name)) for name in top_names]

top_df = pd.DataFrame({'Recipe': top_names_display, 'Similarity Score': top_scores})

plt.figure(figsize=(10, 6))
bars = plt.barh(top_df['Recipe'], top_df['Similarity Score'])
plt.xlabel('Similarity Score')
plt.ylabel('Recipe')
plt.title('Top 10 Recipe Recommendations Based on Previous Orders')

plt.gca().invert_yaxis()

plt.show()